# Competition Certificate Validation — v2 (Fixed)
**CSE-404 · Group B3**

The first version (`B3_competition_certificate_validation.ipynb`) reached 67.4% validation accuracy, but its confusion matrix showed it predicted **every** certificate as *Real*: fake recall was 0. This notebook diagnoses why and fixes it.

| # | Problem in v1 | Fix in v2 |
|---|---|---|
| 1 | Images shrunk to 128×128, which erases forgery artifacts that are only a few pixels wide | Crop the **name, date and logo regions at full resolution** (coordinates from `template_coords.json`) |
| 2 | Training from scratch with 3.3M parameters on 180 images | **Transfer learning**: frozen ImageNet MobileNetV2 as a feature extractor |
| 3 | Class imbalance (2:1) with no correction | **Class weights** (`balanced`) |
| 4 | `horizontal_flip` mirrors text, which never happens on real certificates | Removed |
| 5 | `Dropout` imported but never used | Dropout + L2 in the classifier head |
| 6 | `validation_steps = 46 // 32 = 1`, so only 32 of 46 validation images were ever seen | Whole arrays, no manual step counts |
| 7 | Only accuracy was tracked | Fake recall/precision, macro-F1, ROC-AUC, PR-AUC |
| 8 | Random split: the same certificate template appears in train and validation (leakage) | **5-fold `StratifiedGroupKFold` grouped by template**, so test templates are never seen in training |
| 9 | Single 46-image validation split | Out-of-fold predictions for all images (every image is tested exactly once) |

**Runs on:** Google Colab (Keras 3 + TensorFlow) or locally (Keras 3 + PyTorch backend). The code is backend-independent.

## 1. Setup

In [ ]:
import os, sys, importlib.util
IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB and importlib.util.find_spec('tensorflow') is None:
    os.environ.setdefault('KERAS_BACKEND', 'torch')   # TensorFlow is not available for every Python version

import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from PIL import Image
import keras
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
SEED = 42
keras.utils.set_random_seed(SEED)
print('Keras', keras.__version__, '| backend:', keras.backend.backend())

## 2. Paths
On **Colab**, the Drive folder must contain `dataset/`, `template_coords.json` and the original templates (`AI project/Certificates/`, the folder `generate_certificate_dataset.py` used as input).

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/Pattern DATA SET')
else:
    PROJECT_DIR = Path.cwd()

DATASET_DIR  = PROJECT_DIR / 'dataset'
TEMPLATE_DIR = PROJECT_DIR / 'AI project' / 'Certificates'
COORDS_FILE  = PROJECT_DIR / 'template_coords.json'
MODEL_DIR    = PROJECT_DIR / 'models'

for p in (DATASET_DIR, TEMPLATE_DIR, COORDS_FILE):
    print(f'{"OK " if p.exists() else "MISSING"}  {p}')

## 3. Load the dataset

In [ ]:
IMG_EXT = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
rows = []
for cls, label in (('real', 1), ('fake', 0)):
    for f in sorted((DATASET_DIR / cls).iterdir()):
        if f.suffix.lower() in IMG_EXT:
            rows.append({'path': str(f), 'name': f.name, 'class_label': cls.capitalize(), 'label': label})
df = pd.DataFrame(rows)

meta_path = DATASET_DIR / 'metadata.csv'
if meta_path.exists():
    meta = pd.read_csv(meta_path)
    meta['name'] = meta['filename'].str.split('/').str[-1]
    df = df.merge(meta[['name', 'manipulation_type']], on='name', how='left')
df['manipulation_type'] = df.get('manipulation_type', pd.Series(index=df.index, dtype=object))
df['manipulation_type'] = df['manipulation_type'].fillna(df['label'].map({1: 'none', 0: 'unknown'}))

print(df['class_label'].value_counts().to_string(), '\n')
print(df[df.label == 0]['manipulation_type'].value_counts().to_string())

## 4. Which template did each image come from?
`generate_certificate_dataset.py` writes text onto 23 template images. Each dataset image has the same layout as its template, so we find the template with the smallest pixel difference (on a 64×48 thumbnail).

This matters for two reasons:
1. **Region cropping** needs the template's name/date/logo coordinates.
2. **Leakage-free evaluation**: real and fake images made from the same template look almost identical. If a template appears in both train and test, the model can score well by memorising templates instead of learning forgery artifacts.

In [ ]:
templates = [p for p in sorted(TEMPLATE_DIR.iterdir())
             if p.suffix.lower() in IMG_EXT and not p.name.lower().startswith('_tmp')]
coords = json.loads(COORDS_FILE.read_text(encoding='utf-8'))
templates = [p for p in templates if p.name in coords]
thumb = lambda im: np.asarray(im.convert('RGB').resize((64, 48)), dtype=np.float32)
tpl_thumbs = {p.name: thumb(Image.open(p)) for p in templates}
tpl_sizes = {p.name: Image.open(p).size for p in templates}

def match_template(path):
    t = thumb(Image.open(path))
    dists = {k: float(np.abs(t - v).mean()) for k, v in tpl_thumbs.items()}
    best = min(dists, key=dists.get)
    return best, dists[best]

df[['template', 'template_dist']] = df['path'].apply(lambda p: pd.Series(match_template(p)))
print(f'{len(templates)} templates, {df.template.nunique()} used')
print('Weak matches (distance > 15):', df[df.template_dist > 15]['name'].tolist())
pd.crosstab(df['template'].str[:38], df['class_label'])

## 5. Region extraction (the key fix)
A certificate is about 1000–1600 px wide. Resizing it to 128 px shrinks it ~10×, so a name shifted by 5 px or a small date patch disappears. Instead we crop the three places where `generate_certificate_dataset.py` applies forgeries — **name, date and logo** — from the full-resolution image, and resize each crop to 224×224.

In [ ]:
REGIONS = ['name', 'date', 'logo']

def region_boxes(img_size, tpl_name):
    W, H = img_size
    c = coords[tpl_name]
    tw, th = tpl_sizes[tpl_name]
    sx, sy = W / tw, H / th                     # scale if the image size differs from the template
    nx, ny = c['name_xy']; dx, dy = c['date_xy']
    fs, fd = c['font_size_name'], c['font_size_date']
    boxes = [
        (nx - 30, ny - fs // 2, nx + int(0.55 * tw), ny + int(1.8 * fs)),   # name line
        (dx - 20, dy - fd // 2, dx + int(0.45 * tw), dy + int(1.8 * fd)),   # date line
        tuple(c['logo_box']),                                             # logo
    ]
    out = []
    for x0, y0, x1, y1 in boxes:
        x0, x1 = max(0, int(x0 * sx)), min(W, int(x1 * sx))
        y0, y1 = max(0, int(y0 * sy)), min(H, int(y1 * sy))
        out.append((x0, y0, max(x1, x0 + 8), max(y1, y0 + 8)))
    return out

def region_crops(path, tpl_name, size=224):
    im = Image.open(path).convert('RGB')
    return [np.asarray(im.crop(b).resize((size, size)), dtype=np.float32)
            for b in region_boxes(im.size, tpl_name)]

# Show the regions on one real and one fake certificate from the same template
tpl = df[df.label == 0]['template'].iloc[2]
ex = [df[(df.template == tpl) & (df.label == 1)].iloc[0], df[(df.template == tpl) & (df.label == 0)].iloc[0]]
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for r, row in enumerate(ex):
    im = Image.open(row.path).convert('RGB')
    axes[r, 0].imshow(im)
    for b, col in zip(region_boxes(im.size, row.template), ['red', 'blue', 'green']):
        axes[r, 0].add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=col, lw=2))
    axes[r, 0].set_title(f'{row.class_label} ({row.manipulation_type})')
    for k, crop in enumerate(region_crops(row.path, row.template)):
        axes[r, k + 1].imshow(crop.astype('uint8')); axes[r, k + 1].set_title(REGIONS[k])
for a in axes.ravel(): a.axis('off')
plt.tight_layout(); plt.show()

## 6. Evaluation protocol
- **5-fold `StratifiedGroupKFold`, grouped by template.** Each fold's test templates are never seen during training, and the real:fake ratio is kept similar in every fold.
- **Out-of-fold (OOF) predictions:** every one of the images is predicted exactly once by a model that never saw it, so the metrics use the whole dataset instead of 46 images.
- **Fake is the class we care about.** We report fake recall (how many forgeries we catch), fake precision, macro-F1, ROC-AUC and PR-AUC for the fake class.

In [ ]:
y = df['label'].values
groups = df['template'].values
GROUP_CV  = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
RANDOM_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def metrics(y_true, p_real, thr=0.5):
    pred = (p_real >= thr).astype(int)
    return {
        'accuracy':       accuracy_score(y_true, pred),
        'fake_recall':    recall_score(y_true, pred, pos_label=0, zero_division=0),
        'fake_precision': precision_score(y_true, pred, pos_label=0, zero_division=0),
        'macro_f1':       f1_score(y_true, pred, average='macro', zero_division=0),
        'roc_auc':        roc_auc_score(y_true, p_real),
        'pr_auc_fake':    average_precision_score(1 - y_true, 1 - p_real),
    }

def run_cv(fit_predict, X, cv=GROUP_CV, use_groups=True):
    oof = np.zeros(len(y))
    for k, (tr, te) in enumerate(cv.split(X, y, groups if use_groups else None)):
        keras.utils.set_random_seed(SEED + k)
        oof[te] = fit_predict(X[tr], y[tr], X[te])
    return oof

results, oofs = {}, {}

## 7. Experiment 1 — Reproduce the v1 baseline
Same architecture and settings as v1: 128×128, 3 Conv blocks, Flatten → Dense(128), horizontal flip + 20° rotation, Adam 1e-3, 10 epochs, batch 32, **no class weights**.

In [ ]:
t0 = time.time()
X128 = np.stack([np.asarray(Image.open(p).convert('RGB').resize((128, 128)), dtype=np.float32) / 255.0
                 for p in df['path']])

def build_v1_cnn():
    m = keras.Sequential([
        keras.Input((128, 128, 3)),
        keras.layers.RandomFlip('horizontal'),
        keras.layers.RandomRotation(20 / 360),
        keras.layers.Conv2D(32, 3, activation='relu'), keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation='relu'), keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(128, 3, activation='relu'), keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid'),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def fit_predict_v1(Xtr, ytr, Xte):
    m = build_v1_cnn()
    m.fit(Xtr, ytr, epochs=10, batch_size=32, verbose=0)
    return m.predict(Xte, verbose=0).ravel()

oofs['1. v1 CNN, 128px (baseline)'] = run_cv(fit_predict_v1, X128)
results['1. v1 CNN, 128px (baseline)'] = metrics(y, oofs['1. v1 CNN, 128px (baseline)'])
print(f'{time.time() - t0:.0f}s'); pd.Series(results['1. v1 CNN, 128px (baseline)']).round(3)

## 8. Transfer-learning feature extractor
MobileNetV2 pre-trained on ImageNet (1.28M images), with the classification top removed and global average pooling. It turns any 224×224 image into a 1280-number feature vector. The weights stay **frozen**: with only ~160 training images per fold, we train just a small head on top.

In [ ]:
backbone = keras.applications.MobileNetV2(include_top=False, weights='imagenet',
                                          pooling='avg', input_shape=(224, 224, 3))
backbone.trainable = False
preprocess = keras.applications.mobilenet_v2.preprocess_input   # scales pixels to [-1, 1]

def extract(batch):
    return backbone.predict(preprocess(np.asarray(batch, dtype=np.float32)), batch_size=24, verbose=0)

t0 = time.time()
X_whole = extract([np.asarray(Image.open(p).convert('RGB').resize((224, 224)), dtype=np.float32) for p in df['path']])
crops = np.stack([np.stack(region_crops(p, t)) for p, t in zip(df['path'], df['template'])])   # (N, 3, 224, 224, 3)
X_region3 = extract(crops.reshape(-1, 224, 224, 3)).reshape(len(df), len(REGIONS), -1)          # (N, 3, 1280)
X_region = X_region3.reshape(len(df), -1)                                                       # (N, 3840)
print('whole-image features:', X_whole.shape, '| region features:', X_region.shape, f'| {time.time() - t0:.0f}s')

### Classifier head (shared by Experiments 2 and 3)
- Features are standardised with the **training fold's** mean/std only (no leakage).
- Dropout 0.5 → Dense(64, ReLU, L2) → Dropout 0.3 → Dense(1, sigmoid)
- **Class weights** computed from the training fold.
- **Early stopping** on an inner 20% validation split (restore best weights).

In [ ]:
def build_head(dim):
    inp = keras.Input((dim,))
    x = keras.layers.Dropout(0.5)(inp)
    x = keras.layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-3))(x)
    x = keras.layers.Dropout(0.3)(x)
    out = keras.layers.Dense(1, activation='sigmoid')(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
              metrics=[keras.metrics.AUC(name='auc')])
    return m

def fit_head(Xtr, ytr):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Z = (Xtr - mu) / sd
    tr, va = next(StratifiedShuffleSplit(1, test_size=0.2, random_state=SEED).split(Z, ytr))
    w = compute_class_weight('balanced', classes=np.array([0, 1]), y=ytr[tr])
    m = build_head(Z.shape[1])
    m.fit(Z[tr], ytr[tr], validation_data=(Z[va], ytr[va]), epochs=300, batch_size=32, verbose=0,
          class_weight={0: w[0], 1: w[1]},
          callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)])
    return m, mu, sd

def fit_predict_head(Xtr, ytr, Xte):
    m, mu, sd = fit_head(Xtr, ytr)
    return m.predict((Xte - mu) / sd, verbose=0).ravel()

w = compute_class_weight('balanced', classes=np.array([0, 1]), y=y)
print(f'Class weights on the full data: Fake={w[0]:.2f}, Real={w[1]:.2f}')

## 9. Experiment 2 — Whole image, transfer learning + class weights
All the "standard" fixes (MobileNetV2, 224px, class weights, dropout, no flip), but still looking at the **whole** certificate shrunk to 224×224.

In [ ]:
name = '2. MobileNetV2, whole image 224px'
oofs[name] = run_cv(fit_predict_head, X_whole)
results[name] = metrics(y, oofs[name]); pd.Series(results[name]).round(3)

## 10. Experiment 3 — Region-based model (name + date + logo at full resolution)
The same head, but the input is the concatenated features of the three full-resolution crops (3 × 1280 = 3840).

In [ ]:
name = '3. MobileNetV2, name+date+logo regions'
oofs[name] = run_cv(fit_predict_head, X_region)
results[name] = metrics(y, oofs[name]); pd.Series(results[name]).round(3)

## 11. Results

In [ ]:
res = pd.DataFrame(results).T.round(3)
display(res) if 'display' in globals() else print(res)

ax = res[['fake_recall', 'fake_precision', 'macro_f1', 'roc_auc']].plot.bar(figsize=(11, 4.5), rot=0)
ax.set_ylim(0, 1); ax.set_title('5-fold template-grouped CV (out-of-fold)'); ax.legend(loc='upper left')
ax.set_xticklabels([n.split(',')[0] + '\n' + n.split(',', 1)[1].strip() for n in res.index], fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(oofs), figsize=(5 * len(oofs), 4))
for ax, (n, p) in zip(axes, oofs.items()):
    cm = confusion_matrix(y, (p >= 0.5).astype(int), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(n, fontsize=10)
plt.tight_layout(); plt.show()

best = '3. MobileNetV2, name+date+logo regions'
print(best)
print(classification_report(y, (oofs[best] >= 0.5).astype(int), target_names=['Fake', 'Real'], digits=3))

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
for n, p in oofs.items():
    fpr, tpr, _ = roc_curve(y, p)
    a1.plot(fpr, tpr, label=f'{n.split(".")[0]}. AUC={roc_auc_score(y, p):.2f}')
    pr, rc, _ = precision_recall_curve(1 - y, 1 - p)
    a2.plot(rc, pr, label=f'{n.split(".")[0]}. AP={average_precision_score(1 - y, 1 - p):.2f}')
a1.plot([0, 1], [0, 1], 'k--', lw=1); a1.set(title='ROC (Real = positive)', xlabel='False positive rate', ylabel='True positive rate')
a2.axhline((y == 0).mean(), color='k', ls='--', lw=1, label='chance')
a2.set(title='Precision-Recall for the Fake class', xlabel='Fake recall', ylabel='Fake precision')
a1.legend(); a2.legend(); plt.tight_layout(); plt.show()

## 12. Was the random split hiding a problem? (template leakage check)
Same region model, evaluated with an ordinary random stratified split, where templates **can** appear in both train and test.

In [ ]:
oof_random = run_cv(fit_predict_head, X_region, cv=RANDOM_CV, use_groups=False)
leak = pd.DataFrame({'Random split (templates shared)': metrics(y, oof_random),
                     'Template-grouped split (honest)': results[best]}).T.round(3)
display(leak) if 'display' in globals() else print(leak)

## 13. Error analysis

In [ ]:
df['p_real'] = oofs[best]
df['pred'] = (df['p_real'] >= 0.5).astype(int)
df['correct'] = df['pred'] == df['label']

per_type = (df[df.label == 0].groupby('manipulation_type')['correct']
            .agg(['mean', 'sum', 'count']).rename(columns={'mean': 'fake_recall', 'sum': 'caught', 'count': 'total'}))
print('Fake recall by forgery type:'); print(per_type.round(2).to_string())

# Which single region carries the signal?
region_only = {f'only {r}': metrics(y, run_cv(fit_predict_head, X_region3[:, i])) for i, r in enumerate(REGIONS)}
print(); print(pd.DataFrame(region_only).T.round(3)[['fake_recall', 'macro_f1', 'roc_auc']].to_string())

In [ ]:
wrong = df[~df.correct].sort_values('p_real', key=lambda s: (s - 0.5).abs(), ascending=False).head(8)
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, (_, r) in zip(axes.ravel(), wrong.iterrows()):
    ax.imshow(Image.open(r.path).convert('RGB'))
    ax.set_title(f'{r["name"]}\ntrue={r.class_label} ({r.manipulation_type}) | P(real)={r.p_real:.2f}', fontsize=9)
for ax in axes.ravel(): ax.axis('off')
plt.suptitle('Most confident mistakes'); plt.tight_layout(); plt.show()

### Threshold choice
0.5 is a default, not a law. Lowering the "Real" threshold catches more fakes but rejects more genuine certificates. In a real deployment the threshold should be picked on validation data from the cost of each error.

In [ ]:
rows = []
for thr in (0.3, 0.4, 0.5, 0.6, 0.7):
    m = metrics(y, oofs[best], thr)
    rows.append({'threshold (P(real) ≥ thr → Real)': thr, **{k: round(m[k], 3) for k in ('fake_recall', 'fake_precision', 'macro_f1', 'accuracy')}})
pd.DataFrame(rows)

## 14. Train the final model on all data and save it

In [ ]:
final_head, mu, sd = fit_head(X_region, y)
MODEL_DIR.mkdir(exist_ok=True)
final_head.save(MODEL_DIR / 'certificate_region_head.keras')
np.savez(MODEL_DIR / 'certificate_region_scaler.npz', mean=mu, std=sd)

def predict_certificate(path, template_name=None, thr=0.5):
    # The template layout must be known: it tells us where the name, date and logo are.
    if template_name is None:
        template_name, _ = match_template(path)
    feats = extract(np.stack(region_crops(path, template_name))).reshape(1, -1)
    p_real = float(final_head.predict((feats - mu) / sd, verbose=0)[0, 0])
    return {'template': template_name, 'p_real': round(p_real, 3), 'verdict': 'Real' if p_real >= thr else 'Fake'}

print('Saved to', MODEL_DIR)
for p in (df[df.label == 1].path.iloc[0], df[df.label == 0].path.iloc[0]):
    print(Path(p).name, predict_certificate(p))

## 15. Conclusions
_Filled in below after the run — see the results tables above for the exact numbers of this run._